## Migrated Stage 3. Numerical pilot summaries from the retired prototype have been removed; this notebook writes new `stage3_*` outputs when scans are run.

# Stage 3 — joint threshold experiments in the original AToum model

All six experiments now call the original `model_class.Model` rate dynamics through its `CTCModel` extension. The simulation retains the 40-area anatomy and FLN matrix; W is measured as mean firing rate over F1, F2, F3, F4, 7a, and 7b. The phase oscillator and its long-range gate remain explicit CTC additions.

The former Stage 2 prototype used a separate rate model. Its CSVs and figures have been moved to `legacy_two_node_prototype_results/`; they are not results from this original-model notebook. No Stage 3 scan has been run with this migrated implementation yet.


## Step 0 — imports, parameter window, and scan controls

The baseline uses the Stage 1 stimulus window and ignition rule (W mean rate ≥ 20 Hz for at least 50 ms). K and stimulus values below are candidate settings carried forward from earlier scans, not a validated CTC efficacy interval.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'model_class.py').exists()), None)
if project_root is None:
    raise RuntimeError('Run Jupyter from the repository or a subdirectory.')
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'project_experiment'))
from original_model_runner import run_original_model

OUT = project_root / 'project_experiment'
OUT.mkdir(exist_ok=True)
BASE = {
    'duration_ms': 800.0, 'dt_ms': 0.5,
    'stim_on_ms': 50.0, 'stim_off_ms': 500.0,
    'stimulus_drive': 0.13,  # 130 pA in the original Model
    'g_sw': 0.95, 'phase_gate_strength': 0.5,
    'K_rad_s': 2.5, 'alpha': 0.05,
    'phase_frequency_s_hz': 10.0, 'phase_frequency_w_hz': 10.3,
    'phase_diffusion_rad_sqrt_s': 0.0, 'phase_offset_rad': 0.0,
    'ignition_threshold_hz': 20.0, 'ignition_duration_ms': 50.0,
    'phase_window_ms': 50.0,
    'task_gain': 0.0, 'attention_mode': 'both',
    'decay_mode': 'none', 'task_decay_tau_ms': 100.0,
    'competitor_on_ms': 200.0, 'competitor_level': 1.0,
    'competition_inhibition': 0.8,
    'resource_depletion_per_s': 2.0, 'resource_activity_threshold_hz': 15.0,
    'sensory_area': 'V4', 'workspace_areas': ['F1','F2','F3','F4','7a','7b'],
}
K_WEAK, K_STRONG = 2.0, 3.0
RUN_SCANS = False
N_SEEDS = 5
BASE_SEEDS = np.arange(1, N_SEEDS + 1)
print('Original model entry: model_class.Model + ctc_model.CTCModel')
print('Candidate K=2–3 rad/s; stimulus=120–140 pA; these must be revalidated in this model.')
print('Long scans enabled:', RUN_SCANS, '| seeds per condition:', N_SEEDS)


## Shared simulation model

`g_task` modulates the original model attention channel; `intensity` applies it to sensory input, `synchrony` applies it to phase coupling K, and `both` applies both. `alpha` scales the original local excitatory coupling. Local rates, inhibition, synaptic states, and FLN inputs all use `model_class.Model`.

The simulation returns the W rate, phase difference/PLV, effective attention gain, ignition outcome, and latency. Different mechanisms are exposed as function arguments rather than silently changing the baseline.

In [ ]:
def _wrap_phase(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def _first_ignition(time_ms, rate_hz, threshold_hz, duration_ms, after_ms):
    dt=float(np.median(np.diff(time_ms))); n=max(1,int(np.ceil(duration_ms/dt)))
    valid=(rate_hz>=threshold_hz)&(time_ms>=after_ms)
    if n>len(valid): return False,np.nan,np.nan
    hits=np.flatnonzero(np.convolve(valid.astype(int),np.ones(n,dtype=int),mode='valid')==n)
    if not hits.size: return False,np.nan,np.nan
    idx=int(hits[0]); return True,float(time_ms[idx]-after_ms),float(time_ms[idx])

def phase_lock_acquisition(time_ms,theta_s,theta_w,*,window_ms=100.0,stable_ms=100.0,slope_limit_rad_s=0.75):
    dt=float(np.median(np.diff(time_ms))); win=max(3,int(round(window_ms/dt))); stable=max(1,int(round(stable_ms/dt)))
    delta=np.unwrap(np.asarray(theta_w)-np.asarray(theta_s))
    if len(delta)<win+stable-1: return np.nan
    blocks=np.lib.stride_tricks.sliding_window_view(delta,win); x=(np.arange(win)-(win-1)/2)*dt/1000.0
    slopes=(blocks@x)/np.dot(x,x); valid=np.abs(slopes)<=slope_limit_rad_s
    hits=np.flatnonzero(np.convolve(valid.astype(int),np.ones(stable,dtype=int),mode='valid')==stable)
    return float(time_ms[hits[0]+win-1]) if hits.size else np.nan

def simulate_joint(*,seed=1,params=None,stimulus_schedule=None,attention_mode=None,decay_mode=None,phase_on=True):
    p={**BASE,**(params or {})}
    if attention_mode is not None: p['attention_mode']=attention_mode
    if decay_mode is not None: p['decay_mode']=decay_mode
    result=run_original_model(seed=seed,params=p,configuration='A',stimulus_schedule=stimulus_schedule,phase_on=phase_on)
    onset=float(p.get('ignition_after_ms',p['stim_on_ms']))
    lock=phase_lock_acquisition(result['time_ms'],result['theta_s'],result['theta_w'])
    result['phase_lock_time_ms']=lock
    result['phase_lock_latency_ms']=max(0.0,lock-onset) if np.isfinite(lock) else np.nan
    return result

def summarize_trials(rows):
    frame=pd.DataFrame(rows)
    return frame.groupby(['condition'],dropna=False).agg(n=('seed','count'),ignition_probability=('ignited','mean'),mean_latency_ms=('latency_ms','mean'),latency_variance_ms2=('latency_ms','var'),mean_R=('mean_R','mean'),mean_peak_W_hz=('peak_W_hz','mean')).reset_index()

def one_trial_row(result,condition,seed):
    p=result['parameters']; stim=(result['time_ms']>=p['stim_on_ms'])&(result['time_ms']<p['stim_off_ms']); after=result['time_ms']>=p['stim_off_ms']
    return {'condition':condition,'seed':int(seed),'ignited':result['ignited'],'latency_ms':result['ignition_latency_ms'],'mean_R':float(abs(np.exp(1j*result['delta_phase'][stim]).mean())) if stim.any() else np.nan,'peak_W_hz':float(np.max(result['r_w_hz'])),'mean_W_after_hz':float(result['r_w_hz'][after].mean()) if after.any() else np.nan,'W_above_threshold_after_ms':float(np.sum(result['r_w_hz'][after]>=p['ignition_threshold_hz'])*p['dt_ms'])}

print('Stage 3 runner uses the original Model/CTCModel dynamics.')


## Experiment 3.1 — strength × synchrony threshold map

Scan sensory drive and the **phase-gate setting** (the model knob that makes S→W transmission depend on phase alignment). Compare candidate weak and strong K values (2 and 3 rad/s); Stage 1 did not establish these as a validated CTC interval. The measured PLV is recorded separately; gate strength is not mislabeled as PLV.

In [ ]:
# Exploratory pilot grid; set RUN_SCANS=True above.
if RUN_SCANS:
    drive_grid = np.round(np.arange(0.11, 0.151, 0.01), 2)  # candidate transition range from earlier scans; recheck in this model
    gate_grid = np.linspace(0.0, 1.0, 6)
    rows_21=[]
    from tqdm.auto import tqdm
    progress=tqdm(total=2*len(drive_grid)*len(gate_grid)*len(BASE_SEEDS),desc='Stage 3.1 pilot',unit='run')
    for K in (K_WEAK,K_STRONG):
        for drive in drive_grid:
            for gate in gate_grid:
                for seed in BASE_SEEDS:
                    res=simulate_joint(seed=int(seed),params={**BASE,'K_rad_s':K,'stimulus_drive':drive,'phase_gate_strength':gate})
                    row=one_trial_row(res,'K='+str(K),seed); row.update(stimulus_drive=drive,phase_gate_strength=gate,K_rad_s=K)
                    rows_21.append(row)
                    progress.update(1)
    progress.close()
    exp21=pd.DataFrame(rows_21)
    exp21.to_csv(OUT/'stage3_1_strength_synchrony_trials.csv',index=False)
    map21=exp21.groupby(['K_rad_s','stimulus_drive','phase_gate_strength']).agg(
        ignition_probability=('ignited','mean'),mean_R=('mean_R','mean')).reset_index()
    map21.to_csv(OUT/'stage3_1_bifurcation_summary.csv',index=False)
    display(map21.head())
    fig,axes=plt.subplots(1,2,figsize=(12,4.5),sharey=True)
    fig.suptitle(f'Stage 3.1 scan ({len(BASE_SEEDS)} seeds per grid point)')
    for ax,K in zip(axes,(K_WEAK,K_STRONG)):
        q=map21[map21.K_rad_s==K].pivot(index='phase_gate_strength',columns='stimulus_drive',values='ignition_probability')
        im=ax.imshow(q.to_numpy(),origin='lower',aspect='auto',vmin=0,vmax=1,
                     extent=[q.columns.min(),q.columns.max(),q.index.min(),q.index.max()],cmap='viridis')
        ax.set(title=f'K={K:g} rad/s',xlabel='Stimulus drive',ylabel='Phase-gate setting')
        ax.set_xticks(drive_grid); ax.set_yticks(gate_grid); ax.tick_params(axis='y',labelleft=True)
        fig.colorbar(im,ax=ax,label='W ignition probability')
    fig.tight_layout(); fig.savefig(OUT/'stage3_1_bifurcation_maps.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.1.')


### Stage 3.1 migrated-model results

Not generated yet. The preceding cell writes `stage3_1_*` files when run. Its five-seed setting is for a small exploratory pilot; use at least 50 paired seeds before making a boundary claim.

## Experiment 3.2 — attention gain and boundary shifts

The ablation compares `intensity` (g_task scales stimulus), `synchrony` (g_task scales K), and `both`. The resulting heatmaps show how the ignition boundary moves with task gain. This is a model-defined decomposition, not a claim that biological attention has only these two channels.

In [ ]:
if RUN_SCANS:
    task_grid=np.linspace(0.0,0.5,6); drive_grid_22=np.round(np.arange(0.11,0.151,0.01),2)
    modes=('intensity','synchrony','both'); rows_22=[]
    from tqdm.auto import tqdm
    progress=tqdm(total=len(modes)*len(task_grid)*len(drive_grid_22)*len(BASE_SEEDS),desc='Stage 3.2 pilot',unit='run')
    for mode in modes:
        for gain in task_grid:
            for drive in drive_grid_22:
                for seed in BASE_SEEDS:
                    res=simulate_joint(seed=int(seed),params={**BASE,'K_rad_s':2.5,'stimulus_drive':drive,'task_gain':gain},attention_mode=mode)
                    row=one_trial_row(res,mode,seed); row.update(stimulus_drive=drive,task_gain=gain,attention_mode=mode)
                    rows_22.append(row); progress.update(1)
    progress.close()
    exp22=pd.DataFrame(rows_22); exp22.to_csv(OUT/'stage3_2_attention_ablation_trials.csv',index=False)
    map22=exp22.groupby(['attention_mode','task_gain','stimulus_drive']).agg(
        ignition_probability=('ignited','mean'),mean_R=('mean_R','mean')).reset_index()
    map22.to_csv(OUT/'stage3_2_attention_ablation_summary.csv',index=False)
    fig,axes=plt.subplots(1,3,figsize=(15,4.4),sharey=True)
    for ax,mode in zip(axes,modes):
        q=map22[map22.attention_mode==mode].pivot(index='task_gain',columns='stimulus_drive',values='ignition_probability')
        im=ax.imshow(q.to_numpy(),origin='lower',aspect='auto',vmin=0,vmax=1,
                     extent=[q.columns.min(),q.columns.max(),q.index.min(),q.index.max()],cmap='viridis')
        ax.set(title=mode,xlabel='Stimulus drive',ylabel='g_task')
        ax.set_xticks(drive_grid_22); ax.set_yticks(task_grid); ax.tick_params(axis='y',labelleft=True)
        fig.colorbar(im,ax=ax,label='Ignition probability')
    fig.tight_layout(); fig.savefig(OUT/'stage3_2_attention_ablation.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.2.')


### Stage 3.2 migrated-model results

Not generated yet. The preceding cell writes `stage3_2_*` files when run. The old pilot summary was removed because it came from the retired prototype.

## Experiment 3.3 — recurrent gain α and hysteresis

For each α, the stimulus is ramped up and then down while the network state is carried continuously across plateaus. The onset and offset thresholds are the first/last plateau whose final 30 ms mean W rate crosses the ignition threshold. Hysteresis width = onset threshold − offset threshold. No loop (or no switch-off) is reported as zero/undefined rather than forced to look hysteretic.

In [ ]:
def make_up_down_schedule(levels, dwell_ms=80, dt_ms=1.0):
    up=np.repeat(np.asarray(levels,float),int(round(dwell_ms/dt_ms)))
    down=np.repeat(np.asarray(levels,float)[::-1],int(round(dwell_ms/dt_ms)))
    return np.concatenate([up,down])


def ramp_switch_points(result, levels, dwell_ms=80):
    dt=float(result['parameters']['dt_ms']); n=int(round(dwell_ms/dt)); rates=result['r_w_hz']
    tail=max(1,int(round(30/dt)))
    means_up=[]; means_down=[]
    for j in range(len(levels)):
        means_up.append(float(rates[j*n+n-tail:(j+1)*n].mean()))
        start=(len(levels)+j)*n
        means_down.append(float(rates[start+n-tail:start+n].mean()))
    on=[levels[j] for j,x in enumerate(means_up) if x>=BASE['ignition_threshold_hz']]
    down_levels=np.asarray(levels,float)[::-1]
    off=[down_levels[j] for j,x in enumerate(means_down) if x<BASE['ignition_threshold_hz']]
    onset=min(on) if on else np.nan
    offset=max(off) if off else np.nan
    width=onset-offset if np.isfinite(onset) and np.isfinite(offset) else np.nan
    return onset,offset,width,means_up,means_down

if RUN_SCANS:
    alpha_grid=[0.05,0.2,0.35,0.5,0.7,0.9]
    levels=np.round(np.arange(0.0,0.201,0.01),2); schedule=make_up_down_schedule(levels)
    rows_23=[]
    from tqdm.auto import tqdm
    progress=tqdm(total=len(alpha_grid)*len(BASE_SEEDS),desc='Stage 3.3 pilot',unit='run')
    for alpha in alpha_grid:
        for seed in BASE_SEEDS:
            res=simulate_joint(seed=int(seed),params={**BASE,'alpha':alpha,'stim_on_ms':0.0,'stim_off_ms':1e9,
                'ignition_after_ms':0.0,'stimulus_drive':0.0},stimulus_schedule=schedule)
            on,off,width,_,_=ramp_switch_points(res,levels)
            rows_23.append({'seed':int(seed),'alpha':alpha,'onset_drive':on,'offset_drive':off,'hysteresis_width':width}); progress.update(1)
    progress.close()
    exp23=pd.DataFrame(rows_23); exp23.to_csv(OUT/'stage3_3_hysteresis_trials.csv',index=False)
    hysteresis=exp23.groupby('alpha').agg(mean_onset=('onset_drive','mean'),mean_offset=('offset_drive','mean'),
        mean_width=('hysteresis_width','mean'),sd_width=('hysteresis_width','std')).reset_index()
    hysteresis.to_csv(OUT/'stage3_3_hysteresis_summary.csv',index=False)
    display(hysteresis)
    fig,ax=plt.subplots(figsize=(6,4))
    finite=hysteresis.mean_width.notna()
    lower=np.minimum(hysteresis.loc[finite,'sd_width'].fillna(0),hysteresis.loc[finite,'mean_width'])
    upper=hysteresis.loc[finite,'sd_width'].fillna(0)
    ax.errorbar(hysteresis.loc[finite,'alpha'],hysteresis.loc[finite,'mean_width'],yerr=np.vstack([lower,upper]),marker='o',capsize=3)
    if (~finite).any(): ax.text(.03,.96,'α ≥ 0.5: no switch-off by zero drive',transform=ax.transAxes,va='top',fontsize=9)
    ax.set(xlabel='Local recurrent gain α',ylabel='Hysteresis width (stimulus-drive units)',title='Hysteresis width vs α'); ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(OUT/'stage3_3_hysteresis_vs_alpha.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.3.')


### Stage 3.3 migrated-model results

Not generated yet. The preceding cell writes `stage3_3_*` files when run. The old pilot summary was removed because it came from the retired prototype.

## Experiment 3.4 — ignition latency vs phase-locking speed\n\nHold stimulus above the candidate threshold and compare K values, including a pure intensity-driven control (`phase_on=False`, no phase gate). These K values are exploratory settings, not a validated CTC range. Phase-lock acquisition is defined by a 100-ms unwrapped phase-difference slope below 0.75 rad/s, sustained for 100 ms; this avoids the startup bias of short-window PLV. Report the fraction of trials that lock as well as latency among those that do. This operational threshold should be sensitivity-tested.\n

In [ ]:
if RUN_SCANS:
    delay_K=[0.0,2.0,2.5,3.0]; rows_24=[]
    from tqdm.auto import tqdm
    progress=tqdm(total=len(delay_K)*len(BASE_SEEDS),desc='Stage 3.4 pilot',unit='run')
    for K in delay_K:
        phase_on=K>0
        for seed in BASE_SEEDS:
            gate=0.0 if not phase_on else BASE['phase_gate_strength']
            res=simulate_joint(seed=int(seed),params={**BASE,'stimulus_drive':0.14,'K_rad_s':K,'phase_gate_strength':gate},phase_on=phase_on)
            rows_24.append({'seed':int(seed),'K_rad_s':K,'condition':'coupled' if phase_on else 'no_phase_control',
                'ignited':res['ignited'],'ignition_latency_ms':res['ignition_latency_ms'],
                'phase_lock_latency_ms':res['phase_lock_latency_ms'],'phase_lock_time_ms':res['phase_lock_time_ms'],'mean_R':float(abs(np.exp(1j*res['delta_phase'][50:500]).mean()))}); progress.update(1)
    progress.close()
    exp24=pd.DataFrame(rows_24); exp24.to_csv(OUT/'stage3_4_latency_vs_locking_trials.csv',index=False)
    delay_summary=exp24.groupby(['condition','K_rad_s']).agg(ignition_probability=('ignited','mean'),
        mean_ignition_latency_ms=('ignition_latency_ms','mean'),sd_ignition_latency_ms=('ignition_latency_ms','std'),
        phase_lock_probability=('phase_lock_time_ms',lambda x:x.notna().mean()),mean_lock_latency_ms=('phase_lock_latency_ms','mean'),mean_lock_time_ms=('phase_lock_time_ms','mean'),mean_R=('mean_R','mean')).reset_index()
    delay_summary.to_csv(OUT/'stage3_4_latency_summary.csv',index=False)
    display(delay_summary)
    fig,axes=plt.subplots(1,3,figsize=(14,4.2))
    all_k=sorted(exp24.K_rad_s.unique())
    ignp=exp24.groupby('K_rad_s').ignited.mean().reindex(all_k)
    axes[0].plot(all_k,ignp,'o-'); axes[0].set(xlabel='K (rad/s)',ylabel='Ignition probability',ylim=(-.05,1.05),title='Ignition success')
    coupled=exp24[exp24.condition=='coupled']
    for xk,group in coupled.groupby('K_rad_s'):
        ys=group.ignition_latency_ms.dropna().to_numpy()
        if ys.size:
            jitter=np.linspace(-.045,.045,ys.size) if ys.size>1 else np.array([0.0])
            axes[1].scatter(np.full(ys.size,xk)+jitter,ys,s=24,alpha=.75)
            axes[1].plot([xk-.08,xk+.08],[np.median(ys)]*2,lw=2)
    axes[1].set(xlabel='K (rad/s)',ylabel='Ignition latency (ms; successful trials)',title='Latency (individual seeds)')
    lockp=exp24.groupby('K_rad_s').phase_lock_time_ms.apply(lambda x:x.notna().mean()).reindex(all_k)
    axes[2].plot(all_k,lockp,'o-',color='tab:green'); axes[2].set(xlabel='K (rad/s)',ylabel='Lock-acquisition probability',ylim=(-.05,1.05),title='Stable phase-lock success')
    for ax in axes: ax.grid(alpha=.25)
    fig.tight_layout(); fig.savefig(OUT/'stage3_4_latency_vs_K.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.4.')


### Stage 3.4 migrated-model results

Not generated yet. The preceding cell writes `stage3_4_*` files when run. Phase-lock thresholds should be sensitivity-tested before interpreting latency differences.

## Experiment 3.5 — three attention-decay mechanisms

The scenarios share the same initial gain and task drive: (A) gain decays exponentially after the target reaches ignition; (B) a competitor appears and reduces gain according to the un-suppressed competitor fraction; (C) gain decays with accumulated time spent above ignition threshold. The plotted gain trace makes the implemented mechanism visible.

In [ ]:
if RUN_SCANS:
    decay_scenarios={
        'A_target_decay':{'decay_mode':'target_decay','task_decay_tau_ms':100.0},
        'B_competition':{'decay_mode':'competition','competitor_on_ms':200.0,'competitor_level':2.0,'competition_inhibition':0.3},
        'C_resource':{'decay_mode':'resource','resource_depletion_per_s':2.0},
    }
    rows_25=[]; example_traces={}
    from tqdm.auto import tqdm
    progress=tqdm(total=len(decay_scenarios)*len(BASE_SEEDS),desc='Stage 3.5 pilot',unit='run')
    for label,overrides in decay_scenarios.items():
        for seed in BASE_SEEDS:
            res=simulate_joint(seed=int(seed),params={**BASE,'stimulus_drive':0.14,'task_gain':0.3,**overrides},decay_mode=overrides['decay_mode'])
            row=one_trial_row(res,label,seed); rows_25.append(row)
            if seed==BASE_SEEDS[0]: example_traces[label]=res
            progress.update(1)
    progress.close()
    exp25=pd.DataFrame(rows_25); exp25.to_csv(OUT/'stage3_5_attention_decay_trials.csv',index=False)
    summary25=exp25.groupby('condition').agg(ignition_probability=('ignited','mean'),mean_latency_ms=('latency_ms','mean'),latency_variance_ms2=('latency_ms','var'),mean_W_after_hz=('mean_W_after_hz','mean'),mean_above_threshold_after_ms=('W_above_threshold_after_ms','mean')).reset_index()
    summary25.to_csv(OUT/'stage3_5_attention_decay_summary.csv',index=False)
    display(summary25)
    fig,axes=plt.subplots(2,1,figsize=(9,6),sharex=True)
    for label,res in example_traces.items():
        axes[0].plot(res['time_ms'],res['g_task'],label=label)
        axes[1].plot(res['time_ms'],res['r_w_hz'],label=label)
    axes[0].set_ylabel('Effective g_task'); axes[1].set_ylabel('W rate (Hz)'); axes[1].set_xlabel('Time (ms)')
    axes[1].axhline(BASE['ignition_threshold_hz'],color='k',ls='--',lw=1)
    for ax in axes: ax.grid(alpha=.25); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT/'stage3_5_attention_decay.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.5.')


### Stage 3.5 migrated-model results

Not generated yet. The preceding cell writes `stage3_5_*` files when run. The old pilot summary was removed because it came from the retired prototype.

## Experiment 3.6 — ADHD-inspired parameter perturbations

These are model perturbations, not a clinical diagnosis or a claim that ADHD has one parameter signature. The baseline uses K=3 and task gain 0.25; each perturbation changes one factor: lower task gain, slower phase coupling (K=2, an exploratory candidate), or weaker suppression of a distractor. The combined condition changes all three. For confirmatory estimates, use 50–100 paired seeds and report ignition probability plus latency variance among successful trials.

In [ ]:
if RUN_SCANS:
    adhd_conditions={
        'baseline':{'task_gain':0.25,'K_rad_s':3.0,'competition_inhibition':0.8},
        'lower_task_gain':{'task_gain':0.10,'K_rad_s':3.0,'competition_inhibition':0.8},
        'slower_synchrony':{'task_gain':0.25,'K_rad_s':2.0,'competition_inhibition':0.8},
        'weaker_competition_suppression':{'task_gain':0.25,'K_rad_s':3.0,'competition_inhibition':0.3},
        'combined_perturbation':{'task_gain':0.10,'K_rad_s':2.0,'competition_inhibition':0.3},
    }
    rows_26=[]
    from tqdm.auto import tqdm
    progress=tqdm(total=len(adhd_conditions)*len(BASE_SEEDS),desc='Stage 3.6 pilot',unit='run')
    for label,overrides in adhd_conditions.items():
        for seed in BASE_SEEDS:
            res=simulate_joint(seed=int(seed),params={**BASE,'stimulus_drive':0.12,'competitor_on_ms':200.0,
                'competitor_level':1.0,'decay_mode':'competition',**overrides},decay_mode='competition')
            row=one_trial_row(res,label,seed); row['K_rad_s']=overrides['K_rad_s']; row['task_gain']=overrides['task_gain'];
            row['competition_inhibition']=overrides['competition_inhibition']; rows_26.append(row); progress.update(1)
    progress.close()
    exp26=pd.DataFrame(rows_26); exp26.to_csv(OUT/'stage3_6_adhd_parameter_map_trials.csv',index=False)
    def _latency_var_success(x):
        y=x.dropna().to_numpy(float)
        return float(np.var(y,ddof=1)) if y.size>1 else np.nan
    adhd_summary=exp26.groupby('condition').agg(n=('seed','count'),ignition_probability=('ignited','mean'),
        mean_latency_ms=('latency_ms','mean'),latency_variance_success_ms2=('latency_ms',_latency_var_success),
        mean_peak_W_hz=('peak_W_hz','mean')).reset_index()
    display(adhd_summary)
    adhd_summary.to_csv(OUT/'stage3_6_adhd_parameter_map_summary.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].bar(adhd_summary.condition,adhd_summary.ignition_probability); axes[0].set_ylabel('Ignition probability'); axes[0].tick_params(axis='x',rotation=25)
    axes[1].bar(adhd_summary.condition,adhd_summary.latency_variance_success_ms2); axes[1].set_ylabel('Successful-trial latency variance (ms²)'); axes[1].tick_params(axis='x',rotation=25)
    for ax in axes: ax.grid(axis='y',alpha=.25)
    fig.tight_layout(); fig.savefig(OUT/'stage3_6_adhd_summary.png',dpi=180,bbox_inches='tight'); plt.show()
else:
    print('Set RUN_SCANS=True and run this cell to execute Experiment 3.6.')


### Stage 3.6 migrated-model results

Not generated yet. The preceding cell writes `stage3_6_*` files when run. Use 50–100 paired seeds for confirmatory comparisons; five seeds only preview the code path.

## Interpreting the migrated experiments

Every scan now runs the full original multi-area model. Input values are pA (`stimulus_drive=0.13` means 130 pA); `alpha` scales the original local excitatory coupling relative to its baseline; K and gate depth are parameters of the added CTC extension. The task-gain channel uses the original model's attention multiplier. Target-decay, competition, and resource depletion are implemented as experimental modulators in the CTC subclass.

The previous prototype's pilot CSV/PNG files are not the output of this migrated runner. Run each experiment with `RUN_SCANS=True`, inspect the per-trial files, then update summaries from the new results before drawing conclusions.
